<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Taller Hands-On: El Triage de las Entregas Retrasadas 🚚
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Taller Práctico Evaluativo · Módulo 09
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/homeworks/Para%20Dummies/09_Casos_Estudio_Aplicaciones_Hands_On_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a practicar? 🎈

Esta es la versión **guiada** 🧸 del [taller estándar del Módulo 09](../09_Casos_Estudio_Aplicaciones_Hands_On.ipynb). Vas a recorrer, en pequeño, **el camino de un proyecto de ML de principio a fin**: una empresa de mensajería quiere avisar **qué entregas llegarán tarde**. Repasamos la receta de los cuadernos Para Dummies del módulo ([el triage de urgencias](../../09%20-%20Casos%20de%20Estudio%20y%20Aplicaciones/Para%20Dummies/00_Casos_Estudio_Dummies.ipynb)):

1. 😴 **El guardia que dice «aquí no pasa nada»** (Reto 1): primero, ¿cuánto vale no hacer nada?
2. 🍳 **La receta completa en una tubería** (Reto 2): rellenar huecos, poner todo en la misma escala y entrenar, sin hacer trampa.
3. 💰 **La cuenta que decide a quién llamar** (Reto 3): no es lo mismo equivocarse en un sentido que en el otro.

**Cómo trabajar:** completa los huecos `...` y los `# TODO` (siempre hay una pista 💡), luego ejecuta la celda de **autoverificación**: si algo falta, te dirá qué revisar. Al final, *Restart & Run All*. No necesitas descargar nada.

---
### 📦 Los datos (ejecuta sin modificar)

1 500 entregas **ficticias**. `retraso = 1` significa que la entrega llegó tarde (≈ 20 %). Algunas distancias están sin registrar (casillas vacías).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import recall_score, roc_auc_score
import warnings; warnings.filterwarnings("ignore")

rng = np.random.default_rng(2027)
n = 1500
X = pd.DataFrame({
    "distancia_km": rng.gamma(3.0, 4.0, n).round(1),
    "lluvia_mm": np.where(rng.random(n) < 0.35, rng.exponential(6.0, n), 0.0).round(1),
    "n_paradas": rng.integers(1, 12, n),
    "hora_pico": rng.integers(0, 2, n),
})
eta = -5.2 + 0.14 * X.distancia_km + 0.10 * X.lluvia_mm + 1.0 * X.hora_pico + 0.18 * X.n_paradas
y = pd.Series((rng.random(n) < 1 / (1 + np.exp(-eta))).astype(int), name="retraso")
X.loc[rng.random(n) < 0.08, "distancia_km"] = np.nan
print(X.shape, "| tasa de retraso:", round(y.mean(), 3), "| distancias vacías:", int(X.distancia_km.isna().sum()))
X.head()

---
## Reto 1: ¿Cuánto Vale No Hacer Nada? 😴 *(25 puntos)*

**Analogía.** El guardia que siempre dice «todo tranquilo» acierta casi siempre... y no atrapa a nadie. Esa es tu **línea base**: cualquier modelo debe ganarle.

**Qué hacer.**
1. Separa los datos: 25 % de prueba, `stratify=y`, `random_state=0`.
2. `tasa_retraso`: proporción de entregas tardías en `y_tr`.
3. `acc_dummy`: *accuracy* en prueba del guardia (`DummyClassifier(strategy="most_frequent")`) y `recall_dummy`: su *recall* en prueba (¿cuántas entregas tardías detecta?).

**Meta.** El guardia tiene una *accuracy* cercana a `1 - tasa_retraso` y un *recall* de 0: ¡por eso la *accuracy* sola engaña!

In [ ]:
# 1) Separar
X_tr, X_te, y_tr, y_te = train_test_split(..., ..., test_size=..., stratify=..., random_state=0)   # 💡 completa los huecos

# 2) ¿Qué proporción de entregas tardías hay?
tasa_retraso = None  # TODO: y_tr.mean()

# 3) El guardia: siempre dice "no habrá retraso"
guardia = DummyClassifier(strategy="most_frequent")
guardia.fit(X_tr, y_tr)                     # el Dummy no mira las columnas, solo cuenta clases
acc_dummy = guardia.score(..., ...)         # 💡 evalúalo con la prueba
recall_dummy = recall_score(y_te, guardia.predict(...))   # 💡 predicciones sobre X_te

print("Tasa de retraso:", tasa_retraso, "| accuracy del guardia:", acc_dummy, "| recall del guardia:", recall_dummy)

In [ ]:
# ✅ Autoverificación del Reto 1 (no modifiques esta celda)
_falta = [v for v in ("X_tr", "X_te", "y_tr", "y_te", "tasa_retraso", "acc_dummy", "recall_dummy",) if v not in globals() or globals()[v] is None or globals()[v] is Ellipsis]
assert not _falta, f"💡 Reto 1: todavía faltan por crear/completar: {_falta}. Reemplaza los huecos '...' y los TODO."

assert X_tr.shape == (1125, 4) and X_te.shape == (375, 4), "💡 Reto 1: la partición debe dejar 1125 filas para entrenar y 375 para la prueba (test_size=0.25)."
assert abs(y_tr.mean() - y_te.mean()) < 0.01, "💡 Reto 1: usa stratify=y para que la proporción de retrasos sea parecida en ambos lados."
assert 0.1 < tasa_retraso < 0.3, "💡 Reto 1: tasa_retraso es la proporción de unos en y_tr (≈ 0.2): y_tr.mean()."
assert abs(acc_dummy - (1 - y_te.mean())) < 0.01, "💡 Reto 1: el guardia acierta todo lo que NO es retraso; su accuracy debe ser ≈ 1 - tasa de retraso. Evalúalo con X_te, y_te."
assert recall_dummy == 0, "💡 Reto 1: el guardia nunca dice 'retraso', así que su recall es 0. Calcúlalo con guardia.predict(X_te)."
print("✅ Reto 1 superado: ya sabes el piso que hay que superar.")

---
## Reto 2: La Receta Completa en una Tubería 🍳 *(25 puntos)*

**Analogía.** Una receta tiene pasos que **siempre se hacen en el mismo orden**: (1) rellenar los huecos, (2) poner todo en la misma escala, (3) cocinar el modelo. Una `Pipeline` guarda la receta completa y garantiza que, en cada ronda de validación, los pasos 1 y 2 se «aprendan» **solo con la parte de entrenamiento**: nada de espiar el examen.

**Qué hacer.**
1. Arma `tuberia` con `Pipeline` y tres pasos: `("imputador", SimpleImputer(strategy="median"))`, `("escalador", StandardScaler())` y `("clf", LogisticRegression())`.
2. Con `cross_val_score(..., cv=5, scoring="roc_auc")` calcula `auc_cv` (el promedio) de `tuberia` sobre `X_tr`, `y_tr` y `auc_cv_base` (el del guardia `DummyClassifier(strategy="prior")` metido en una tubería con el mismo imputador).
3. Ajusta `tuberia` con `X_tr`, `y_tr` y calcula `auc_test` sobre la prueba con las probabilidades `tuberia.predict_proba(X_te)[:, 1]`.

**Meta.** El guardia saca AUC = 0.5; tu tubería debe superar 0.7 tanto en validación como en la prueba, y las dos cifras deben parecerse (no hubo trampa).

In [ ]:
# 1) La receta: imputar -> escalar -> modelo
tuberia = Pipeline([
    ("imputador", SimpleImputer(strategy=...)),   # 💡 "median"
    ("escalador", StandardScaler()),
    ("clf", LogisticRegression()),
])

# 2) Puntaje con validación cruzada (5 rondas), solo con el entrenamiento
auc_cv = cross_val_score(tuberia, ..., ..., cv=5, scoring="roc_auc").mean()    # 💡 datos y etiquetas de entrenamiento
tuberia_base = Pipeline([("imputador", SimpleImputer(strategy="median")), ("clf", DummyClassifier(strategy="prior"))])
auc_cv_base = None  # TODO: lo mismo con tuberia_base

# 3) Examen final sellado
tuberia.fit(X_tr, y_tr)
p_te = tuberia.predict_proba(...)[:, 1]      # 💡 probabilidad de retraso en la prueba
auc_test = None  # TODO: roc_auc_score(y_te, p_te)

print("AUC CV guardia:", auc_cv_base, "| AUC CV tubería:", auc_cv, "| AUC prueba:", auc_test)

In [ ]:
# ✅ Autoverificación del Reto 2 (no modifiques esta celda)
_falta = [v for v in ("tuberia", "auc_cv", "auc_cv_base", "p_te", "auc_test",) if v not in globals() or globals()[v] is None or globals()[v] is Ellipsis]
assert not _falta, f"💡 Reto 2: todavía faltan por crear/completar: {_falta}. Reemplaza los huecos '...' y los TODO."

assert isinstance(tuberia, Pipeline) and [n for n, _ in tuberia.steps] == ["imputador", "escalador", "clf"], "💡 Reto 2: la tubería debe tener los pasos 'imputador', 'escalador' y 'clf' en ese orden."
assert isinstance(tuberia.steps[0][1], SimpleImputer) and tuberia.steps[0][1].strategy == "median", "💡 Reto 2: el imputador es SimpleImputer(strategy='median')."
assert abs(auc_cv_base - 0.5) < 0.02, "💡 Reto 2: el guardia 'prior' debe tener AUC ≈ 0.5 con cross_val_score(tuberia_base, X_tr, y_tr, cv=5, scoring='roc_auc')."
assert auc_cv > 0.7, "💡 Reto 2: tu tubería debería superar 0.7 de AUC en validación cruzada; revisa que uses X_tr, y_tr y scoring='roc_auc'."
assert len(p_te) == 375 and 0 <= p_te.min() and p_te.max() <= 1, "💡 Reto 2: p_te son 375 probabilidades (tuberia.predict_proba(X_te)[:, 1])."
assert auc_test > 0.7, "💡 Reto 2: el AUC en la prueba debería superar 0.7: roc_auc_score(y_te, p_te)."
assert abs(auc_test - auc_cv) < 0.08, "💡 Reto 2: el AUC de validación y el de la prueba deberían parecerse (± 0.08); si no, hubo trampa o mala suerte."
print("✅ Reto 2 superado: tu receta funciona y no hizo trampa.")

---
## Reto 3: La Cuenta que Decide a Quién Llamar 💰 *(25 puntos)*

**Analogía.** En urgencias, mandar a casa a alguien grave cuesta muchísimo más que revisar a alguien sano. Aquí: una entrega tardía **sin avisar** (falso negativo) cuesta **15**, y llamar **sin necesidad** (falso positivo) cuesta **2**. Con esa cuenta, el umbral 0.5 ya no es el mejor.

**Qué hacer.**
1. Completa `costo(y_real, y_pred)` $= 15\cdot FN + 2\cdot FP$.
2. Calcula `costos_val`: el costo sobre el **entrenamiento** (probabilidades fuera de muestra `p_val` ya calculadas abajo) para cada umbral de la lista `umbrales`.
3. `umbral_opt` es el umbral con el menor costo. Mide en la **prueba**: `costo_opt` (con `umbral_opt`), `costo_05` (con 0.5) y `costo_nada` (nunca avisar: todo 0).

**Meta.** `umbral_opt` debe quedar por debajo de 0.5 y debe cumplirse `costo_opt < costo_05 < costo_nada`.

In [ ]:
C_FN, C_FP = 15, 2

def costo(y_real, y_pred):
    """Costo total = 15 * falsos negativos + 2 * falsos positivos."""
    y_real, y_pred = np.asarray(y_real), np.asarray(y_pred)
    fn = np.sum((y_real == 1) & (y_pred == 0))
    fp = np.sum((y_real == 0) & (y_pred == 1))
    return None  # TODO: C_FN * fn + C_FP * fp

# Probabilidades "fuera de muestra" del entrenamiento (ya resuelto)
p_val = cross_val_predict(tuberia, X_tr, y_tr, cv=5, method="predict_proba")[:, 1]

umbrales = np.linspace(0.02, 0.9, 45)
costos_val = [costo(y_tr, (p_val >= u).astype(int)) for u in umbrales]
umbral_opt = umbrales[...]          # 💡 la posición del costo mínimo: np.argmin(costos_val)

# Examen final sellado
costo_opt = costo(y_te, (p_te >= umbral_opt).astype(int))
costo_05 = None      # TODO: lo mismo pero con umbral 0.5
costo_nada = None    # TODO: costo de no avisar nunca: costo(y_te, np.zeros(len(y_te), dtype=int))
print("Umbral óptimo:", umbral_opt, "| costo en prueba -> óptimo:", costo_opt, "| 0.5:", costo_05, "| nunca avisar:", costo_nada)

In [ ]:
# ✅ Autoverificación del Reto 3 (no modifiques esta celda)
_falta = [v for v in ("umbral_opt", "costo_opt", "costo_05", "costo_nada",) if v not in globals() or globals()[v] is None or globals()[v] is Ellipsis]
assert not _falta, f"💡 Reto 3: todavía faltan por crear/completar: {_falta}. Reemplaza los huecos '...' y los TODO."

assert costo(np.array([1, 1, 0, 0]), np.array([0, 1, 1, 0])) == 17, "💡 Reto 3: costo([1,1,0,0], [0,1,1,0]) debe ser 15 (un falso negativo) + 2 (un falso positivo) = 17."
assert costo(np.array([1, 0]), np.array([1, 0])) == 0, "💡 Reto 3: sin errores el costo es 0."
assert np.isclose(umbral_opt, umbrales[int(np.argmin(costos_val))]), "💡 Reto 3: umbral_opt = umbrales[np.argmin(costos_val)]."
assert 0 < umbral_opt < 0.5, f"💡 Reto 3: como un falso negativo es mucho más caro, el umbral óptimo debe ser menor que 0.5 (obtuviste {umbral_opt})."
assert costo_nada == 15 * int(y_te.sum()), "💡 Reto 3: nunca avisar cuesta 15 por cada entrega tardía de la prueba."
assert abs(costo_05 - costo(y_te, (p_te >= 0.5).astype(int))) < 1e-9, "💡 Reto 3: costo_05 es el costo en la prueba con umbral 0.5."
assert costo_opt < costo_05 < costo_nada, f"💡 Reto 3: debe cumplirse costo_opt < costo_05 < costo_nada; obtuviste {costo_opt}, {costo_05}, {costo_nada}."
print(f"✅ Reto 3 superado: el umbral por costo ahorra {costo_05 - costo_opt} unidades frente a 0.5.")

---
## Reto 4: Mini-Reflexión ✍️ *(25 puntos)*

Escribe **3 o 4 líneas** con tus palabras (sin código): ¿por qué la *accuracy* del guardia engaña? ¿Qué ventaja tiene meter el rellenado y el escalado **dentro** de la tubería? ¿Por qué el mejor umbral no fue 0.5 y qué cambiaría si llamar sin necesidad costara 10 en vez de 2? 🧸

#### ✍️ Tu respuesta

_Escribe aquí tus 3 o 4 líneas. Reemplaza este texto por tu respuesta._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: partición y línea base (el guardia) | 25 |
| Reto 2: tubería sin fuga, validada y probada | 25 |
| Reto 3: umbral por costo frente a 0.5 y a no avisar | 25 |
| Reto 4: mini-reflexión con tus palabras | 25 |

### ✅ Checklist de entrega

- [ ] No quedan `...` ni `# TODO` sin completar.
- [ ] Las tres autoverificaciones imprimen «✅ Reto N superado».
- [ ] Hiciste *Restart & Run All* y todo corrió sin errores.

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>